# ForecastEx

## Web REST API

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests, json, os
from pprint import pprint

## Live markets

In [3]:
from get_live_markets import get_live_markets
markets = get_live_markets()

In [4]:
from add_category_to_markets import add_category_to_markets
df_markets = add_category_to_markets(markets)
df_markets.head(3)

,category,name,symbol,conid
0,Climate,Annual Global Temperature Threshold,GTTA,776339242
1,Climate,Atmospheric Carbon Dioxide,ACD,712856705
2,Climate,Global Carbon Dioxide Emissions,GCE,732764729


In [5]:
markets[0]

{'symbol': 'MNYCG',
 'conid': 796056051,
 'name': 'General Election for New York City Mayor'}

In [6]:
conid_to_market = {market['conid']: market for market in markets}

In [7]:
conid_to_market[776339242]

{'symbol': 'GTTA',
 'conid': 776339242,
 'name': 'Annual Global Temperature Threshold'}

## Get each contract of a market

In [8]:
from fetch_all_contracts import fetch_all_contracts
from tqdm.auto import tqdm

In [9]:
for conid in tqdm(conid_to_market):
    conid_to_market[conid]['contracts'] = fetch_all_contracts(conid)

  0%|          | 0/203 [00:00<?, ?it/s]

## Get current probability for each market contract

In [10]:
from get_candidate_probability import get_candidate_probability

In [ ]:
for market in tqdm(conid_to_market.values()):
    for contract in tqdm(market['contracts']):
        if 'probability' in contract:
            continue
        try:
            contract['probability'] = get_candidate_probability(contract['conid'])
        except:
            print("Fail on", contract['conid'])

## Open Interest (must run chrome in debug mode)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [53]:
all_conids = [contract['conid']
              for market in conid_to_market.values()
              for contract in market['contracts']]

In [54]:
len(all_conids)

14720

In [58]:
from run_get_OI import run_get_OI

In [64]:
conids = await run_get_OI(all_conids)

In [80]:
len(conids)

2703

In [71]:
def OI_to_integer(OI):
    if OI[-1] == 'K':
        return float(OI[:-1])*1000.0
    elif OI[-1] == 'M':
        return float(OI[:-1])*1000000.0
    else:
        return float(OI)

In [73]:
conid_to_OI = {int(conid): OI_to_integer(OI) for conid, OI in conids.items()}

In [78]:
for market in conid_to_market.values():
    for contract in market['contracts']:
        try:
            contract['OI'] = conid_to_OI[contract['conid']]
        except:
            pass

In [79]:
fn = 'forecastex.json'
with open(fn, 'w') as f:
    json.dump(conid_to_market, f, indent=4)

## Filter for contracts with OI

In [88]:
ctx_with_OI = {}
for market in conid_to_market:
    mkt = conid_to_market[market]
    for contract in mkt['contracts']:
        if 'OI' in contract:
            if market not in ctx_with_OI:
                ctx_with_OI[market] = mkt
                ctx_with_OI[market]['contracts'] = []
            ctx_with_OI[market]['contracts'].append(contract)

## Filter for contracts with OI and probability

In [104]:
liquid_ctx = {}
for market in ctx_with_OI:
    mkt = ctx_with_OI[market]
    for contract in mkt['contracts']:
        if contract['probability']:
            if market not in liquid_ctx:
                liquid_ctx[market] = mkt
                liquid_ctx[market]['contracts'] = []
            liquid_ctx[market]['contracts'].append(contract)

In [109]:
fn = 'liquid_ctx.json'
with open(fn, 'w') as f:
    json.dump(liquid_ctx, f, indent=4)

In [110]:
!ls -l {fn}

-rw-rw-r-- 1 catskills catskills 5287705 Jul 23 20:07 liquid_ctx.json


## How many available binary bets for real y'alls

In [112]:
sum([len(mkt['contracts']) for mkt in liquid_ctx.values()])/2

420.0

## In a table

In [147]:
bets = []
for market in liquid_ctx.values():
    for contract in market['contracts']:
        row = {x: market[x] for x in ['symbol', 'name']}
        row['conid'] = contract['conid']
        prob = contract['probability']
        row['forecast'] = prob['probability_pct']
        row['volume'] = prob['volume'][-1]
        row['OI'] = contract['OI']
        row['longDescription'] = contract['longDescription']
        row['yesNo'] = 'YES' if contract['putOrCall'] == 'C' else 'NO'
        bets.append(row)

In [ ]:
import pandas as pd
from add_category_to_markets import add_category_to_markets
df0 = add_category_to_markets(pd.DataFrame(bets))

In [173]:
df0.category.unique()

array(['Macro', 'Event', 'Rates', 'Climate', 'Index', 'Election',
       'Energy', 'Crypto', 'FX'], dtype=object)

In [174]:
df0[df0.category == 'Election']

,symbol,name,conid,forecast,volume,OI,longDescription,yesNo,category
368,PCD,US Presidential Candidate Democrat,745924025,26.0,0.0,4610.0,Will Gavin Newsom win the Democratic Nomination for US President in 2028?,YES,Election
369,PCD,US Presidential Candidate Democrat,745924028,75.0,0.0,4610.0,Will Gavin Newsom win the Democratic Nomination for US President in 2028?,NO,Election
370,PCD,US Presidential Candidate Democrat,745924033,9.0,0.0,71.0,Will Josh Shapiro win the Democratic Nomination for US President in 2028?,YES,Election
371,PCD,US Presidential Candidate Democrat,745924036,92.0,0.0,71.0,Will Josh Shapiro win the Democratic Nomination for US President in 2028?,NO,Election
372,PCD,US Presidential Candidate Democrat,745924048,5.0,0.0,40.0,Will Kamala Harris win the Democratic Nomination for US President in 2028?,YES,Election
373,PCD,US Presidential Candidate Democrat,745924053,96.0,0.0,40.0,Will Kamala Harris win the Democratic Nomination for US President in 2028?,NO,Election
374,PCD,US Presidential Candidate Democrat,745924067,7.0,0.0,27.0,Will Alexandria Ocasio-Cortez win the Democratic Nomination for US President in 2028?,YES,Election
375,PCD,US Presidential Candidate Democrat,745924070,94.0,0.0,27.0,Will Alexandria Ocasio-Cortez win the Democratic Nomination for US President in 2028?,NO,Election
376,PCR,US Presidential Candidate Republican,745924112,58.0,0.0,5360.0,Will JD Vance win the Republican Nomination for US President in 2028?,YES,Election
377,PCR,US Presidential Candidate Republican,745924117,43.0,0.0,5360.0,Will JD Vance win the Republican Nomination for US President in 2028?,NO,Election


## Combine YES and NO rows

In [176]:
import pandas as pd
df = df0
# Example dataframe: df
# Columns: 'category', 'name', 'longDescription', 'yesNo', 'conid', 'forecast', 'volume', 'OI'

# Step 1: Split the dataframe into YES and NO rows
df_yes = df[df['yesNo'] == 'YES'].copy()
df_no = df[df['yesNo'] == 'NO'].copy()

# Step 2: Rename the relevant columns in each split
df_yes = df_yes.rename(columns={
    'forecast': 'YES',
    'conid': 'conidYes',
    'volume': 'volume',
    'OI': 'OI'
})

df_no = df_no.rename(columns={
    'forecast': 'NO',
    'conid': 'conidNo'
})

# Step 3: Drop unnecessary columns from each before merge
df_yes = df_yes[['category', 'name', 'longDescription', 'YES', 'volume', 'OI', 'conidYes']]
df_no = df_no[['category', 'name', 'longDescription', 'NO', 'conidNo']]

# Step 4: Merge YES and NO dataframes on 'category', 'name', 'longDescription'
df3 = pd.merge(df_yes, df_no, on=['category', 'name', 'longDescription'])

# Resulting dataframe
# Columns: 'category', 'name', 'longDescription', 'YES', 'volume', 'OI', 'conidYes', 'NO', 'conidNo'

# (Optional) Rearranging columns
df3 = df3[['category', 'name', 'longDescription', 'YES', 'NO', 'volume', 'OI', 'conidYes', 'conidNo']]

df3

,category,name,longDescription,YES,NO,volume,OI,conidYes,conidNo
0,Macro,US Fed Funds Target Rate,"Will the US Fed Funds Target Rate be set above 4.375% at the FOMC meeting ending July 30, 2025?",2.0,99.0,0.0,1680.0,722489526,722489529
1,Macro,US Fed Funds Target Rate,"Will the US Fed Funds Target Rate be set above 3.375% at the FOMC meeting ending October 28, 2026?",33.0,68.0,0.0,20.0,722489580,722489586
2,Macro,US Fed Funds Target Rate,"Will the US Fed Funds Target Rate be set above 4.125% at the FOMC meeting ending October 29, 2025?",32.0,69.0,0.0,257.0,722489591,722489592
3,Macro,US Fed Funds Target Rate,"Will the US Fed Funds Target Rate be set above 4.125% at the FOMC meeting ending December 10, 2025?",17.0,84.0,0.0,232.0,722489623,722489624
4,Macro,Social Security Retirement Age,Will Congress enact an increase in the retirement age for Social Security before the end of 2028?,39.0,62.0,0.0,122.0,723652281,723652284
5,Macro,Singapore Real GDP,Will the annualized growth rate in Singapore Real GDP exceed 2.3% in Q2 2025?,99.0,2.0,0.0,490.0,725468905,725468908
6,Event,US Recession,Will the United States economy enter a recession by the end of Q2 2025?,7.0,94.0,0.0,195000.0,726746811,726746814
7,Event,US Recession,Will the United States economy enter a recession by the end of Q3 2025?,21.0,80.0,0.0,560.0,726746819,726746822
8,Event,US Recession,Will the United States economy enter a recession by the end of Q4 2025?,28.0,73.0,0.0,330.0,726746834,726746839
9,Macro,US Top Marginal Income Tax,Will the US top marginal income tax rate exceed 37% for tax year 2026?,38.0,63.0,0.0,145.0,729193794,729193799


In [180]:
df3[df3.category == 'Election']

,category,name,longDescription,YES,NO,volume,OI,conidYes,conidNo,Spread
17,Election,US Presidential Candidate Democrat,Will Gavin Newsom win the Democratic Nomination for US President in 2028?,26.0,75.0,0.0,4610.0,745924025,745924028,49.0
18,Election,US Presidential Candidate Democrat,Will Josh Shapiro win the Democratic Nomination for US President in 2028?,9.0,92.0,0.0,71.0,745924033,745924036,83.0
19,Election,US Presidential Candidate Democrat,Will Kamala Harris win the Democratic Nomination for US President in 2028?,5.0,96.0,0.0,40.0,745924048,745924053,91.0
20,Election,US Presidential Candidate Democrat,Will Alexandria Ocasio-Cortez win the Democratic Nomination for US President in 2028?,7.0,94.0,0.0,27.0,745924067,745924070,87.0
21,Election,US Presidential Candidate Republican,Will JD Vance win the Republican Nomination for US President in 2028?,58.0,43.0,0.0,5360.0,745924112,745924117,15.0
22,Election,US Presidential Candidate Republican,Will Glenn Youngkin win the Republican Nomination for US President in 2028?,4.0,97.0,0.0,287.0,745924152,745924157,93.0
82,Election,Kansas Governor Democratic Primary,Will Julie Holscher win the Kansas Democratic primary for governor in 2026?,39.0,62.0,0.0,290.0,767285263,767285265,23.0
83,Election,New York Governor Democratic Primary,Will Kathy Hochul win the New York Democratic primary for governor in 2026?,71.0,30.0,0.0,124.0,767285565,767285568,41.0
84,Election,New York Governor Democratic Primary,Will Ritchie Torres win the New York Democratic primary for governor in 2026?,9.0,92.0,0.0,85.0,767285573,767285578,83.0
85,Election,New York Governor Republican Primary,Will Mike Lawler win the New York Republican primary for governor in 2026?,8.0,93.0,0.0,430.0,767285583,767285585,85.0


## Filter for centered probabilities

In [177]:
df3['Spread'] = (df3['YES']-df3['NO']).abs()

In [190]:
df3[(df3.Spread < 30) & (df3.longDescription.str.contains('2025')) & (df3.OI > 1000) & df3.longDescription.apply(lambda x: 'December' not in x and 'FY' not in x)].sort_values(by=['category', 'name']) 

,category,name,longDescription,YES,NO,volume,OI,conidYes,conidNo,Spread
119,Index,US 500 Forecast Contract,"Will Dec-25 CME E-mini S&P 500 Index Futures settle above $6,450 on September 30 2025?",52.0,49.0,1000.0,3150.0,773081538,773081543,3.0
195,Index,US Consumer Price Index Yearly,Will the year-over-year change in the US Consumer Price Index exceed 2.7% in July 2025?,63.0,38.0,0.0,16900.0,787978480,787978485,25.0
28,Macro,US Fed Funds Target Rate,"Will the US Fed Funds Target Rate be set above 4.125% at the FOMC meeting ending September 17, 2025?",46.0,55.0,0.0,2880.0,747748569,747748572,9.0


## Filter for multiple bins with dominant probability